# Probability — AI Engineering Interview Prep

Covers: distributions, Bayes theorem, CLT, Monte Carlo, sampling.

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use("seaborn-v0_8")
print("Libraries loaded.")

## 1. Discrete Distributions

In [ ]:
# Binomial — n trials, p success probability
n, p = 20, 0.3
binom = stats.binom(n=n, p=p)
k = np.arange(0, n+1)
print(f"Binomial B({n},{p}): mean={binom.mean():.2f}, std={binom.std():.2f}")
print(f"P(X=6) = {binom.pmf(6):.4f}")
print(f"P(X<=8) = {binom.cdf(8):.4f}")

# Poisson — events per unit time (lambda = avg rate)
lam = 3.5   # avg 3.5 calls/hour
poisson = stats.poisson(mu=lam)
print(f"\nPoisson(λ={lam}): mean={poisson.mean()}, std={poisson.std():.3f}")
print(f"P(X=5) = {poisson.pmf(5):.4f}")
print(f"P(X>=7) = {1 - poisson.cdf(6):.4f}")

## 2. Continuous Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Normal
x = np.linspace(-4, 4, 300)
for mu, sigma, label in [(0,1,"N(0,1)"), (1,0.5,"N(1,0.5)"), (-1,2,"N(-1,2)")]:
    axes[0].plot(x, stats.norm.pdf(x, mu, sigma), label=label)
axes[0].set_title("Normal PDFs")
axes[0].legend()

# Exponential
x = np.linspace(0, 5, 300)
for scale in [0.5, 1, 2]:
    axes[1].plot(x, stats.expon.pdf(x, scale=scale), label=f"λ={1/scale:.1f}")
axes[1].set_title("Exponential PDFs")
axes[1].legend()

# CDF of Normal
x = np.linspace(-4, 4, 300)
axes[2].plot(x, stats.norm.cdf(x))
axes[2].axvline(1.96, color="r", linestyle="--", label="z=1.96 (97.5%)")
axes[2].axhline(0.975, color="r", linestyle=":")
axes[2].set_title("Normal CDF")
axes[2].legend()

plt.tight_layout()
plt.show()

# Key percentiles
norm = stats.norm()
print(f"\n68-95-99.7 rule:")
print(f"P(-1<Z<1) = {norm.cdf(1) - norm.cdf(-1):.4f}")
print(f"P(-2<Z<2) = {norm.cdf(2) - norm.cdf(-2):.4f}")
print(f"P(-3<Z<3) = {norm.cdf(3) - norm.cdf(-3):.4f}")
print(f"95th pct  = {norm.ppf(0.95):.3f}  (z-score)")

## 3. Bayes Theorem

In [ ]:
# Medical test example
# Disease prevalence, test sensitivity and specificity
P_disease   = 0.01    # 1% prevalence
sensitivity = 0.99    # P(positive | disease)
specificity = 0.95    # P(negative | healthy)

P_healthy           = 1 - P_disease
P_pos_given_disease = sensitivity
P_pos_given_healthy = 1 - specificity   # false positive rate

# Total probability P(positive)
P_positive = (P_pos_given_disease * P_disease +
              P_pos_given_healthy * P_healthy)

# Bayes: P(disease | positive)
P_disease_given_pos = (P_pos_given_disease * P_disease) / P_positive

print(f"Disease prevalence:       {P_disease*100:.1f}%")
print(f"Test sensitivity:         {sensitivity*100:.1f}%")
print(f"Test specificity:         {specificity*100:.1f}%")
print(f"P(positive):              {P_positive*100:.2f}%")
print(f"P(disease | +ve test):    {P_disease_given_pos*100:.2f}%  ← surprisingly low!")
print(f"\nKey insight: Low base rate => many false positives dominate")

## 4. Law of Large Numbers

In [ ]:
# Coin flip convergence
flips = np.random.choice([0, 1], size=10000)
cumulative_means = np.cumsum(flips) / np.arange(1, len(flips)+1)

plt.figure(figsize=(10, 4))
plt.plot(cumulative_means, label="Running mean")
plt.axhline(0.5, color="r", linestyle="--", label="True mean (0.5)")
plt.fill_between(range(len(cumulative_means)),
                 0.5 - 1/np.sqrt(np.arange(1, len(flips)+1)),
                 0.5 + 1/np.sqrt(np.arange(1, len(flips)+1)),
                 alpha=0.2, label="±1/√n band")
plt.xlabel("Number of flips")
plt.ylabel("P(Heads)")
plt.title("Law of Large Numbers: Coin Flip Convergence")
plt.legend()
plt.show()

## 5. Central Limit Theorem

In [ ]:
# Exponential population (very non-normal)
population = np.random.exponential(scale=2, size=100000)
sample_sizes = [1, 5, 30, 100]

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, n in zip(axes, sample_sizes):
    sample_means = [np.mean(np.random.choice(population, n)) for _ in range(2000)]
    ax.hist(sample_means, bins=40, density=True, color="steelblue", alpha=0.7)
    # Overlay normal curve predicted by CLT
    mu_c = population.mean()
    se = population.std() / np.sqrt(n)
    x = np.linspace(min(sample_means), max(sample_means), 200)
    ax.plot(x, stats.norm.pdf(x, mu_c, se), "r-", lw=2)
    ax.set_title(f"n={n}")
    ax.set_xlabel("Sample mean")

plt.suptitle("Central Limit Theorem: Exponential Population → Normal Sample Means")
plt.tight_layout()
plt.show()

## 6. Monte Carlo Estimation

In [ ]:
# Estimate π by throwing random darts
n_darts = 100_000
x = np.random.uniform(-1, 1, n_darts)
y = np.random.uniform(-1, 1, n_darts)
inside = (x**2 + y**2) <= 1
pi_estimate = 4 * inside.sum() / n_darts
print(f"π estimate: {pi_estimate:.5f}  (true: {np.pi:.5f})  error: {abs(pi_estimate - np.pi):.5f}")

# Monte Carlo integration: estimate ∫₀¹ x² dx = 1/3
x_sample = np.random.uniform(0, 1, 100_000)
integral_estimate = np.mean(x_sample**2)   # E[f(X)] ≈ (1/(b-a)) ∫f(x)dx
print(f"∫₀¹ x² dx ≈ {integral_estimate:.5f}  (true: {1/3:.5f})")

## Interview Tips

- **Bayes pitfall**: with low prevalence, even a 99% accurate test has many false positives (base rate neglect).
- **CLT**: holds for n≥30 in practice (often); requires finite variance; enables z-tests/t-tests.
- **Exponential distribution**: memoryless — P(X>s+t|X>s) = P(X>t). Used for arrival times, failure rates.
- **Normal percentiles to know**: z=1.645 (90%), z=1.96 (95%), z=2.576 (99%).
- **Monte Carlo**: power is in integration, risk estimation, simulation. Convergence ∝ 1/√n.